In [99]:
# Import necessary packages
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.v2 as transformsV2
from PIL import Image

# Import packages for training and semi-supervised learning
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision.datasets import DatasetFolder

# For the progress bar
from tqdm.auto import tqdm

In [100]:
# Define transforms (data augmentation) for training data
train_tfm = transformsV2.Compose([
    transformsV2.Resize((128, 128)), # Resize images into a fixed shape, which is width = hight = 128
    # Data augmentation
    transformsV2.AutoAugment(), # Use image net data augmentation policies
    transformsV2.ToImage(),
    transformsV2.ToDtype(torch.float32, scale=True),
])

# Do not need augmentations in test and validations
test_tfm = transformsV2.Compose([
    transformsV2.Resize((128, 128)),
    transformsV2.ToImage(),
    transformsV2.ToDtype(torch.float32, scale=True),
])

In [101]:
batch_size = 128

# Construct datasets
# The parameter transform tells the DatasetFolder how to read the data
train_set = DatasetFolder("food-11/training/labeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
unlabeled_set = DatasetFolder("food-11/training/unlabeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
validation_set = DatasetFolder("food-11/validation", loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)
test_set = DatasetFolder("food-11/testing", lambda x: Image.open(x), extensions="jpg", transform=test_tfm)

# Construct dataloaders
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
validation_loader = DataLoader(validation_set, batch_size=batch_size, num_workers=16, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

In [102]:
from torchvision.models import resnet50

# Define model architecture
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # input image size: [3, 128, 128]
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0), # feature map size: [64, 64, 64]
            
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0), # feature map size: [128, 32, 32]
            
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0) # feature map size: [256, 16, 16]
        )
        
        self.fc_layers = nn.Sequential(
            nn.Linear(256 * 16 * 16, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 11)
        )
        
    def forward(self, x):
        # Extract features in images by convolutional layers
        x = self.cnn_layers(x)
        
        # The extracted feature map must be flatten before going to fully-connected layers
        x = x.flatten(1)
        
        # Use fully-connected layers to learn and obtian final logits
        output = self.fc_layers(x)
        
        return output

In [103]:
# Define a pseudo dataset
class PseudoDataset(Dataset):
    def __init__(self, dataset, indices, pseudo_labels):
        self.dataset = dataset
        self.indices = indices
        self.pseudo_labels = pseudo_labels
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        img, _ = self.dataset[self.indices[idx]]
        return img, self.pseudo_labels[idx]

In [104]:
# Define a function to get pseudo labels for unlabeled data, which is necessary in semi-supervised learning
def get_pseudo_labels(dataset, model, threshold=0.65):
    # This function returns an instance of torchvision.datasets.DatasetFolder containing images whose prediction confidences exceed a given threshold
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Construct a dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    model.eval() # Set the model as evaluation mode
    softmax = nn.Softmax(dim=-1)
    
    selected_indices = []
    pseudo_labels = []
    start_idx = 0 # start index of current batch
    
    for batch in tqdm(dataloader):
        imgs, _ = batch
        with torch.no_grad():
            logits = model(imgs.to(device))
        probs = softmax(logits)
        max_probs, preds = torch.max(probs, dim=1)
        for i in range(len(max_probs)):
            if max_probs[i] >= threshold:
                selected_indices.append(start_idx + i)
                pseudo_labels.append(preds[i].item())
        
        # Update start index of next batch
        start_idx += len(imgs)
                
    model.train() # Set the model back to training mode
                
    return PseudoDataset(dataset, selected_indices, pseudo_labels)

In [105]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Classifier().to(device)
model.device = device

# Use cross-entropy as loss
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-5)

num_epochs = 50

do_semi = True

In [106]:
# Train
for epoch in range(num_epochs):
    if do_semi:
        # Get a pseudo dataset with pseudo labels and concat it with the training set
        pseudo_set = get_pseudo_labels(unlabeled_set, model, threshold=0.6)
        concated_set = ConcatDataset([train_set, pseudo_set])
        train_loader = DataLoader(concated_set, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
        
    model.train()
    
    train_loss = []
    train_accs = []
    
    for batch in tqdm(train_loader):
        imgs, labels = batch
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        optimizer.zero_grad() # Clear previous gradients
        loss.backward() # Calculate gradients
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=10) # Clip the gradient norms for stabel training, prevent exploding gradients
        optimizer.step() # Update weights
        
        acc = (logits.argmax(dim=-1) == labels).float().mean()       
        train_loss.append(loss.item())
        train_accs.append(acc)
        
    avg_train_loss = sum(train_loss) / len(train_loss)
    avg_train_acc = sum(train_accs) / len(train_accs)
    
    print(f"[Train | {epoch + 1:03d}/{num_epochs:03d}] loss = {avg_train_loss:.5f}, acc = {avg_train_acc:.5f}")
    
    # Validation
    model.eval()
    
    validation_loss = []
    validation_accs = []
    
    for batch in tqdm(validation_loader):
        imgs, labels = batch
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            logits = model(imgs)
            loss = criterion(logits, labels)
            
            acc = (logits.argmax(dim=-1) == labels).float().mean()
            validation_loss.append(loss.item())
            validation_accs.append(acc)
            
    avg_validation_loss = sum(validation_loss) / len(validation_loss)
    avg_validation_acc = sum(validation_accs) / len(validation_accs)
    
    print(f"[Validation | {epoch + 1:03d}/{num_epochs:03d}] loss = {avg_validation_loss:.5f}, acc = {avg_validation_acc:.5f}")

100%|██████████| 25/25 [00:03<00:00,  6.65it/s]


[Train | 001/050] loss = 2.44043, acc = 0.11719


100%|██████████| 6/6 [00:01<00:00,  4.92it/s]


[Validation | 001/050] loss = 2.55400, acc = 0.10417


100%|██████████| 25/25 [00:04<00:00,  6.24it/s]


...

[Validation | 049/050] loss = 2.15526, acc = 0.49323


100%|██████████| 63/63 [00:07<00:00,  8.44it/s]


[Train | 050/050] loss = 0.68036, acc = 0.78283


100%|██████████| 6/6 [00:01<00:00,  4.85it/s]

[Validation | 050/050] loss = 1.74685, acc = 0.49948


In [58]:
# Test
model.eval()

predictions = []

for batch in tqdm(test_loader):
    imgs, _ = batch
    imgs = imgs.to(device)
    with torch.no_grad():
        logits = model(imgs)
        
    # Take the class with greatest logit as prediction and record it
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

100%|██████████| 27/27 [00:08<00:00,  3.06it/s]


In [59]:
# Save predictions into a file
with open("prediction.csv", 'w') as f:
    f.write("Id,Category\n")
    for i, pred in enumerate(predictions):
        f.write(f"{i},{pred}\n")